In [ ]:
import torch
import numpy as np
from darts import TimeSeries
from pandas import Timedelta

from aare.fetching.AareDataset import AareDataset
from aare.params import read_params
from aare.preparation import (
    resample,
    remove_faulty_periods_aare_temp,
    remove_outliers_aare_temp,
    interpolate_aare_temp,
)
from aare.fetching.remote_existenz_store import RemoteExistenzStore
from aare.darts_utils import to_ts

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
params = read_params()
store = RemoteExistenzStore()
ds = AareDataset.from_conf()

In [ ]:
df = ds.get_val()
o_df = df.copy()
df

In [ ]:
df = resample(df)
df = remove_faulty_periods_aare_temp(df)
df = remove_outliers_aare_temp(df)
df = interpolate_aare_temp(df, drop_filled=True)

In [ ]:
df

In [ ]:
# must avoid NaNs, for real training use
# better imputation methods (or partial training).
# df = df.ffill()
# Instead of this, split into multiple series at the gaps (see below).

In [ ]:
ts = to_ts(df)
ts

In [ ]:
ts.gaps().sort_values("gap_size", ascending=False)

In [ ]:
import darts.utils.missing_values

# split at the remaining gaps (instead of filling them up)
tss = darts.utils.missing_values.extract_subseries(ts)
len(tss)

In [ ]:
season = 24
horizon = 4 * 24

In [ ]:
from darts.models import NaiveSeasonal, GlobalNaiveSeasonal

snaive = NaiveSeasonal(K=24)  # daily seasonality
snaive

In [ ]:
# test every day always with a horizon of 4 days using as information the 24 hours prior.
historical_forecasts = snaive.historical_forecasts(
    tss,
    stride=season,
    train_length=season,
    forecast_horizon=horizon,
    last_points_only=False,
)

print(len(historical_forecasts))
print(type(historical_forecasts[0]))
print(len(historical_forecasts[0]))

In [ ]:
from darts.metrics import mae, rmse

# MAE and RMSE are prob the most appropriate metrics (see eda and -> specs).
backtest = snaive.backtest(tss, historical_forecasts=historical_forecasts, metric=[mae, rmse])
backtest

In [ ]:
np.mean(backtest, axis=0)

In [ ]:
# last_forecast: TimeSeries = historical_forecasts[-1]
last_forecast: TimeSeries = historical_forecasts[-1][-1]
ts[last_forecast.start_time() - Timedelta(24, "h") : last_forecast.end_time()].plot(label="actual")
last_forecast.plot(label="forecast")

In [ ]:
# If we don't use darts for forecasting: historical_forecasts is just a list of TS with forecast values (length of the horizon)
# so we can construct this ourselves and then still use the backtest function of darts to easily calculate the metrics.
# I think that should work. But hopefully there's a good global darts model that already works with the non-retrain mode of backtest.
# for global models, the train_length would be replaced by input_chunk_length on the model itself, I assume.
# Then you could use pre-trained mode (aka non-retrain).

In [ ]:
snaive_g = GlobalNaiveSeasonal(input_chunk_length=season, output_chunk_length=1)
snaive_g

In [ ]:
snaive_g.fit(tss)

In [ ]:
historical_forecasts = snaive_g.historical_forecasts(
    tss,
    stride=season,
    forecast_horizon=horizon,
    last_points_only=False,
    retrain=False,
)
print(len(historical_forecasts))
print(type(historical_forecasts[0]))
print(len(historical_forecasts[0]))

In [ ]:
from darts.metrics import mae, rmse

backtest = snaive_g.backtest(tss, historical_forecasts=historical_forecasts, metric=[mae, rmse])
backtest

In [ ]:
np.mean(backtest, axis=0)

In [ ]:
historical_forecasts[-1][-1]

In [ ]:
# last_forecast: TimeSeries = historical_forecasts[-1]
last_forecast: TimeSeries = historical_forecasts[-1][-1]
ts[last_forecast.start_time() - Timedelta(24, "h") : last_forecast.end_time()].plot(label="actual")
last_forecast.plot(label="forecast")